In [ ]:
yeonu@gpusystem:/nas/arpa_h_repository/public_data/CRC_Atlas_1/CRC-Atlas-Split/crc-atlas/Deg_data/DEG_H5AD_Original/result$ ls
B_cell_Wilcoxon_DEG_crcatlas.csv           Mini_10percent_Wilcoxon_DEG_crcatlas.csv  Schwann_cell_Wilcoxon_DEG_crcatlas.csv
Cancer_cell_Wilcoxon_DEG_crcatlas.csv      Myeloid_cell_Wilcoxon_DEG_crcatlas.csv    Stromal_cell_Wilcoxon_DEG_crcatlas.csv
Epithelial_cell_Wilcoxon_DEG_crcatlas.csv  Neutrophil_Wilcoxon_DEG_crcatlas.csv      T_cell_Wilcoxon_DEG_crcatlas.csv
ILC_Wilcoxon_DEG_crcatlas.csv              NK_Wilcoxon_DEG_crcatlas.csv
Mast_cell_Wilcoxon_DEG_crcatlas.csv        Plasma_cell_Wilcoxon_DEG_crcatlas.csv

In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1. DEG_Wilcoxon 전처리 
# Wilcoxon_DEG : Wilcoxon_DEG_~.csv

df_wilcoxon = pd.read_csv('/nas/arpa_h_repository/public_data/CRC_Atlas_1/CRC-Atlas-Split/crc-atlas/Deg_data/DEG_H5AD_Original/result/Myeloid_cell_Wilcoxon_DEG_crcatlas.csv')
print(df_wilcoxon.shape)
print(df_wilcoxon.head())
df_wilcoxon = df_wilcoxon[["names", "logfoldchanges", "pvals", "pvals_adj"]]
df_wilcoxon.rename(columns={"names": "gene","pvals": "pval","pvals_adj": "Wilcoxon_adj_pval"}, inplace=True)
print(df_wilcoxon.head())

(28476, 5)
   names     scores  logfoldchanges  pvals  pvals_adj
0  RPL30  135.44653        1.042582    0.0        0.0
1  RPL28  134.98103        1.125629    0.0        0.0
2  RPL29  130.39635        1.009054    0.0        0.0
3   RPS7  128.68733        0.984268    0.0        0.0
4  RPL19  127.71720        1.000787    0.0        0.0
    gene  logfoldchanges  pval  Wilcoxon_adj_pval
0  RPL30        1.042582   0.0                0.0
1  RPL28        1.125629   0.0                0.0
2  RPL29        1.009054   0.0                0.0
3   RPS7        0.984268   0.0                0.0
4  RPL19        1.000787   0.0                0.0


In [3]:
df_wilcoxon.head(2)

,gene,logfoldchanges,pval,Wilcoxon_adj_pval
0,RPL30,1.042582,0.0,0.0
1,RPL28,1.125629,0.0,0.0


## <font color="red">**데이터프레임**

In [4]:
import pandas as pd
from functools import reduce

# 1) 미리 각 df에서 gene + adj_pval 컬럼만 추출
dfs = [
    df_wilcoxon[["gene", "logfoldchanges", "pval", "Wilcoxon_adj_pval"]],
    # df_limmatrend[["gene", "limmatrend_adj_pval"]],
    # df_RISCQP[["gene", "RISCQP_adj_pval"]],
    # df_MAST[["gene", "MAST_adj_pval"]],
    # df_monocle_filtered[["gene", "monocle_adj_pval"]],
]

# 2) Reduce를 이용해 순차적으로 outer merge
df_mmm = reduce(
    lambda left, right: pd.merge(left, right, on="gene", how="outer"),
    dfs
)

# 결과 확인
print(df_mmm.shape)
print(df_mmm.head())

(28476, 4)
    gene  logfoldchanges  pval  Wilcoxon_adj_pval
0  RPL30        1.042582   0.0                0.0
1  RPL28        1.125629   0.0                0.0
2  RPL29        1.009054   0.0                0.0
3   RPS7        0.984268   0.0                0.0
4  RPL19        1.000787   0.0                0.0


In [5]:
df_mmm.head(2)

,gene,logfoldchanges,pval,Wilcoxon_adj_pval
0,RPL30,1.042582,0.0,0.0
1,RPL28,1.125629,0.0,0.0


In [6]:
# 1) 각 컬럼별 rank 계산 (값이 작을수록 1위, NaN은 가장 마지막)
df_mmm['rank_wilcoxon'] = df_mmm['Wilcoxon_adj_pval'] \
    .rank(method='min', ascending=True, na_option='bottom').astype(int)

# df_mmm['rank_limma'] = df_mmm['limmatrend_adj_pval'] \
#     .rank(method='min', ascending=True, na_option='bottom').astype(int)

# df_mmm['rank_RISCQP'] = df_mmm['RISCQP_adj_pval'] \
#     .rank(method='min', ascending=True, na_option='bottom').astype(int)

# df_mmm['rank_MAST'] = df_mmm['MAST_adj_pval'] \
#     .rank(method='min', ascending=True, na_option='bottom').astype(int)


# 2) 5개 rank의 합 계산
df_mmm['rank_sum'] = df_mmm[
    ['rank_wilcoxon', #'rank_limma', 'rank_RISCQP', 'rank_MAST'
     ]
].sum(axis=1)

# 3) rank_sum 기준으로 정렬
df_mmm = df_mmm.sort_values('rank_sum').reset_index(drop=True)

# 결과 확인 (선택사항)
df_mmm

,gene,logfoldchanges,pval,Wilcoxon_adj_pval,rank_wilcoxon,rank_sum
0,CSRNP1,-0.705329,0.000000,0.0,1,1
1,FOXO3,-0.762117,0.000000,0.0,1,1
2,SELENOK,-0.450885,0.000000,0.0,1,1
3,SELENOP,-1.150901,0.000000,0.0,1,1
4,LAIR1,-0.539012,0.000000,0.0,1,1
...,...,...,...,...,...,...
28471,SPA17,0.363560,0.405125,1.0,11472,11472
28472,BTK,0.147392,0.404662,1.0,11472,11472
28473,ENSG00000284773,0.451014,0.404093,1.0,11472,11472
28474,PI16,3.237153,0.403799,1.0,11472,11472


In [7]:
# df_mmm_re = df_mmm[['rank_sum', 'gene', 'rank_wilcoxon', #'rank_limma', 'rank_RISCQP', 'rank_MAST'
#                     ]]
# df_mmm_re.head(20)

In [8]:
# 원하는 순서의 리스트
genes_of_interest = ["VSIG4", "LYVE1", "CLEC5A", "CX3CR1", "FN1", "IL11", "NRAP", "SDC1", "SCG5"]
df_mmm_filtered = df_mmm[df_mmm["gene"].isin(genes_of_interest)]
df_mmm_filtered

,gene,logfoldchanges,pval,Wilcoxon_adj_pval,rank_wilcoxon,rank_sum
206,VSIG4,-0.931835,0.000000e+00,0.000000e+00,1,1
575,CX3CR1,1.455885,4.517300e-308,2.233240e-306,576,576
1496,FN1,-0.718652,5.098386e-97,9.698171e-96,1497,1497
3135,LYVE1,-1.237154,5.044069e-26,4.579665e-25,3135,3135
3280,CLEC5A,-0.057228,1.056871e-23,9.172652e-23,3281,3281
9005,SCG5,0.805294,6.714150e-02,2.122942e-01,9006,9006
9720,SDC1,-0.736024,1.460725e-01,4.278752e-01,9721,9721
12555,NRAP,-0.530428,9.886685e-01,1.000000e+00,11472,11472
17752,IL11,-0.603548,8.226040e-01,1.000000e+00,11472,11472


In [9]:
import pandas as pd
from functools import reduce

# 1️⃣ 각 메서드별 “gene + sign” DataFrame 생성
sign_dfs = []

# Wilcoxon: logfoldchanges >0 → MSI, <0 → MSS
sign_dfs.append(
    df_mmm_filtered[['gene','logfoldchanges']]
    .assign(Wilcoxon_sign=lambda d: d['logfoldchanges'].gt(0).map({True:'MSI', False:'MSS'}))
    [['gene','Wilcoxon_sign']]
)

# limmatrend: log2FC <0 → MSI, >0 → MSS
# sign_dfs.append(
#     df_limmatrend[['gene','logfoldchanges']]
#     .assign(limmat_sign=lambda d: d['logfoldchanges'].lt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','limmat_sign']]
# )

# RISCQP: log2FC <0 → MSI, >0 → MSS
# sign_dfs.append(
#     df_RISCQP[['gene','log2FC']]
#     .assign(RISCQP_sign=lambda d: d['log2FC'].lt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','RISCQP_sign']]
# )

# monocle: log2FC >0 → MSI, <0 → MSS
# sign_dfs.append(
#     df_monocle[['gene','log2FC']]
#     .assign(monocle_sign=lambda d: d['log2FC'].gt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','monocle_sign']]
# )

# # MAST: logfoldchanges >0 → MSI, <0 → MSS
# sign_dfs.append(
#     df_MAST[['gene','logfoldchanges']]
#     .assign(MAST_sign=lambda d: d['logfoldchanges'].gt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','MAST_sign']]
# )

# MAST: logfoldchanges >0 → MSI, <0 → MSS
# sign_dfs.append(
#     df_MAST[['gene','logfoldchanges']]
#     .assign(MAST_sign=lambda d: d['logfoldchanges'].lt(0).map({True:'MSI', False:'MSS'}))
#     [['gene','MAST_sign']]
# )

# 2️⃣ 모든 sign DataFrame outer merge
sign_df = reduce(lambda a,b: pd.merge(a,b,on='gene',how='outer'), sign_dfs)

# 3️⃣ rank 합계 df와 결합
df_final = df_mmm_filtered.merge(sign_df, on='gene', how='left')

# 4️⃣ consensus: MSI_sign이 3개 이상 → MSI_high, else MSS_high
# sign_cols = [col for col in df_final.columns if col.endswith('_sign')]
# df_final['consensus'] = df_final[sign_cols].apply(
#     lambda row: 'MSI_high' if (row=='MSI').sum() >= 3 else 'MSS_high',
#     axis=1
# )

# 5️⃣ 결과 확인 (상위 20개)
print(df_final.sort_values('rank_sum').head(20))

     gene  logfoldchanges           pval  Wilcoxon_adj_pval  rank_wilcoxon  \
0   VSIG4       -0.931835   0.000000e+00       0.000000e+00              1   
1  CX3CR1        1.455885  4.517300e-308      2.233240e-306            576   
2     FN1       -0.718652   5.098386e-97       9.698171e-96           1497   
3   LYVE1       -1.237154   5.044069e-26       4.579665e-25           3135   
4  CLEC5A       -0.057228   1.056871e-23       9.172652e-23           3281   
5    SCG5        0.805294   6.714150e-02       2.122942e-01           9006   
6    SDC1       -0.736024   1.460725e-01       4.278752e-01           9721   
7    NRAP       -0.530428   9.886685e-01       1.000000e+00          11472   
8    IL11       -0.603548   8.226040e-01       1.000000e+00          11472   

   rank_sum Wilcoxon_sign  
0         1           MSS  
1       576           MSI  
2      1497           MSS  
3      3135           MSS  
4      3281           MSS  
5      9006           MSI  
6      9721           M

In [10]:
df_final

,gene,logfoldchanges,pval,Wilcoxon_adj_pval,rank_wilcoxon,rank_sum,Wilcoxon_sign
0,VSIG4,-0.931835,0.000000e+00,0.000000e+00,1,1,MSS
1,CX3CR1,1.455885,4.517300e-308,2.233240e-306,576,576,MSI
2,FN1,-0.718652,5.098386e-97,9.698171e-96,1497,1497,MSS
3,LYVE1,-1.237154,5.044069e-26,4.579665e-25,3135,3135,MSS
4,CLEC5A,-0.057228,1.056871e-23,9.172652e-23,3281,3281,MSS
5,SCG5,0.805294,6.714150e-02,2.122942e-01,9006,9006,MSI
6,SDC1,-0.736024,1.460725e-01,4.278752e-01,9721,9721,MSS
7,NRAP,-0.530428,9.886685e-01,1.000000e+00,11472,11472,MSS
8,IL11,-0.603548,8.226040e-01,1.000000e+00,11472,11472,MSS


In [11]:
import numpy as np

# 1) 기존 neg_log10_adj_pval 계산
epsilon = np.nextafter(0, 1)
df_final['neg_log10_adj_pval'] = -np.log10(
    df_final['Wilcoxon_adj_pval'].replace(0, epsilon)
)

# 2) 지수표현(예: 2.44e+02)으로 소수점 둘째 자리까지 포맷
df_final['neg_log10_adj_pval'] = df_final['neg_log10_adj_pval'] \
    .apply(lambda x: f"{x:.1e}")

# 3) gene 순서 재배열
desired_order = ["VSIG4", "LYVE1", "CLEC5A", "CX3CR1",
                 "FN1", "IL11", "NRAP", "SDC1", "SCG5"]
df_final = (
    df_final
    .set_index('gene')
    .reindex(desired_order)
    .reset_index()
)

# 확인
print(df_final[['gene', 'neg_log10_adj_pval', 'Wilcoxon_sign']])

     gene neg_log10_adj_pval Wilcoxon_sign
0   VSIG4            3.2e+02           MSS
1   LYVE1            2.4e+01           MSS
2  CLEC5A            2.2e+01           MSS
3  CX3CR1            3.1e+02           MSI
4     FN1            9.5e+01           MSS
5    IL11           -0.0e+00           MSS
6    NRAP           -0.0e+00           MSS
7    SDC1            3.7e-01           MSS
8    SCG5            6.7e-01           MSI
